# Hello, welcome to the "How to Train Your Own Neural Network" tutorial. 
I'm Isabella, and I should be around the room to help you figure things out. 

## Convolutional Neural Networks

We started our short lecture by describing that a convolutional neural network is a specialized type of deep learning algorithm primarily designed to recognize objects through image classification, object detection, and segmentation (as a subset of the capabilities in a field called computer vision). 

Let's get started by importing the libraries that we will need for this tutorial:

In [6]:
import torch 


## Images as tensors

- A **tensor** is a block of numbers laid out along one or more directions, called **axes**
- Its **shape** lists how long each axis is

Here is an example of a tensor that we manually input the matrix values of:


In [ ]:
M = torch.tensor([[1, 3, 2, 0],
                  [5, 2, 1, 1],
                  [0, 1, 8, 6],
                  [2, 4, 3, 7]], dtype=torch.float32)

print(F.max_pool2d(M[None, None], kernel_size=2, stride=2)[0, 0].int().tolist())

NameError: name 'F' is not defined

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

import numpy as np, torch
import matplotlib.pyplot as plt
from matplotlib import font_manager
from PIL import Image, ImageOps

DATA = Path("data")
(DATA / "fonts").mkdir(parents=True, exist_ok=True)                # Urbanist, SIL Open Font Licence
for style in ("Regular", "Bold"):
    ttf = DATA / "fonts" / f"Urbanist-{style}.ttf"
    if not ttf.exists():
        urlretrieve(f"https://raw.githubusercontent.com/coreyhu/Urbanist/main/fonts/ttf/{ttf.name}", ttf)
    font_manager.fontManager.addfont(ttf)
plt.rcParams["font.family"] = ["Urbanist", "DejaVu Sans"]

full = ImageOps.fit(Image.open(DATA / "oxford-iiit-pet/images/Abyssinian_119.jpg").convert("RGB"),
                    (480, 480), Image.LANCZOS, centering=(0.5, 0.38))

fig, axes = plt.subplots(1, 3, figsize=(13, 5))
for ax, side, label in zip(axes, (480, 64, 24), ("High resolution", "Low resolution", "The 24 × 24 grid")):
    image = full if side == 480 else full.resize((side, side), Image.LANCZOS)
    ax.imshow(image, interpolation="nearest")                   # nearest: show every pixel as a square
    ax.set_title(label, loc="left", weight="bold")
    ax.set_xlabel(f"{side} × {side} pixels  ·  {3 * side * side:,} numbers", fontsize=11)
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

In [ ]:
row, col = 340, 180                                                # a spot of orange fur on the chest
rgb = np.asarray(full)                                             # every pixel's red, green and blue, 0 to 255
print(f"{type(rgb)}   shape {rgb.shape}   dtype {rgb.dtype}   rows × columns × (red, green, blue)")
print(f"the pixel at row {row}, column {col}   red {rgb[row, col, 0]}   green {rgb[row, col, 1]}   blue {rgb[row, col, 2]}")

fig, axes = plt.subplots(1, 3, figsize=(13, 5))
for ax, k, name in zip(axes, range(3), ("Red", "Green", "Blue")):
    channel = np.zeros_like(rgb)
    channel[..., k] = rgb[..., k]                                  # keep one colour, set the other two to 0
    ax.imshow(channel)
    ax.plot(col, row, "o", mfc="none", mec="white", ms=14, mew=2)
    ax.set_title(f"{name} values", loc="left", weight="bold")
    ax.set_xlabel(f"480 × 480 numbers  ·  {rgb[row, col, k]} inside the circle", fontsize=11)
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

x = torch.tensor(rgb).permute(2, 0, 1)                             # the same numbers as a tensor, colours first
print(f"{type(x)}   shape {tuple(x.shape)}   dtype {x.dtype}   (red, green, blue) × rows × columns\n")
print("3 × 3 pixels from the circle onwards: one small matrix per colour")
print(x[:, row:row + 3, col:col + 3])

In [ ]:
N, ZY, ZX, ZS = 24, 10, 6, 5                                       # grid, then the zoom window
small = full.resize((N, N), Image.LANCZOS)                         # the same cat, highly pixelated
px    = np.asarray(small)                                          # colour: red, green, blue per pixel
grey  = np.asarray(small.convert("L"))                             # black and white: one number per pixel

fig, (left, right) = plt.subplots(1, 2, figsize=(13, 6.9))
left.imshow(full)
left.add_patch(plt.Rectangle((ZX * 20 - .5, ZY * 20 - .5), ZS * 20, ZS * 20,   # 20 photo pixels per grid square
                             fill=False, ec="crimson", lw=2.2))
left.set_title("A picture of a cat", loc="left", weight="bold")
left.set_xlabel("480 × 480 pixels", fontsize=11)

right.imshow(grey, cmap="gray", vmin=0, vmax=255)
for i in range(N):
    for j in range(N):
        right.text(j, i, grey[i, j], ha="center", va="center", fontsize=7,
                   color="black" if grey[i, j] > 110 else "white")
right.add_patch(plt.Rectangle((ZX - .5, ZY - .5), ZS, ZS, fill=False, ec="crimson", lw=2.2))
right.set_title(f"What the computer sees: a {N} × {N} matrix", loc="left", weight="bold")
right.set_xlabel("0 is black, 255 is white", fontsize=11)

for ax in (left, right):
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

bw     = torch.tensor(grey)                                        # the same numbers, as PyTorch tensors
colour = torch.tensor(px).permute(2, 0, 1)                         # colours first, the order PyTorch expects
print(f"black and white   {type(bw)}   shape {tuple(bw.shape)}       rows × columns")
print(f"colour            {type(colour)}   shape {tuple(colour.shape)}    3 colours × rows × columns\n")
print("the top-left 2 × 2 pixels of the red box: one small matrix per colour")
print(colour[:, ZY:ZY + 2, ZX:ZX + 2])

In [ ]:
patch = px[ZY:ZY + ZS, ZX:ZX + ZS]                              # the red box, (5, 5, 3)

fig, axes = plt.subplots(1, 4, figsize=(14, 4.2))
for k, cmap in ((2, "Blues"), (1, "Greens"), (0, "Reds")):      # back to front
    axes[0].imshow(patch[..., k], cmap=cmap, vmin=0, vmax=255, zorder=3 - k,
                   extent=(2 * k, 2 * k + ZS, ZS - k, -k))
axes[0].set_xlim(-.4, ZS + 4.4); axes[0].set_ylim(ZS + .4, -2.4)
axes[0].set_title("3 matrices, stacked", loc="left", weight="bold")

for ax, k, cmap in zip(axes[1:], range(3), ("Reds", "Greens", "Blues")):
    ax.imshow(patch[..., k], cmap=cmap, vmin=0, vmax=255)
    for i in range(ZS):
        for j in range(ZS):
            ax.text(j, i, patch[i, j, k], ha="center", va="center", fontsize=12,
                    color="w" if patch[i, j, k] > 165 else "black")
    ax.set_title(f"{'RGB'[k]} channel", loc="left", weight="bold")

for ax in axes:
    ax.set_xticks([]); ax.set_yticks([])
    ax.spines[:].set_visible(False)
fig.suptitle(f"Zoomed in: one pixel = {tuple(patch[1, 1])},   the patch = 3 × {ZS} × {ZS} "
             f"= {3 * ZS * ZS} integers", x=.01, y=1.02, ha="left", fontsize=12)
plt.tight_layout(); plt.show()

In [2]:
IMG_SIZE = 176


## Labels: 0 is cat, 1 is dog

## Convolution, one step at a time

**What `F.conv2d` does**

- A **kernel** is a small grid of weights, here 3 × 3. This one looks for vertical edges: negative weights on the left, positive on the right
- Lay it over a 3 × 3 window of the image, multiply each pixel by the weight on top of it, and add up the nine products
- That one number goes into the **feature map**. Then the kernel moves one pixel along and does it again
- A window 3 pixels wide fits 24 − 3 + 1 = 22 times across the image, so the feature map is 22 × 22: 484 steps

**Step 1, by hand**

- The top-left window is background, nearly flat grey: 209, 212, 215 / 208, 213, 216 / 212, 214, 219
- The kernel's middle column is 0, so only the left and right columns count

$$
(-1)(209) + (1)(215) \;+\; (-2)(208) + (2)(216) \;+\; (-1)(212) + (1)(219) \;=\; 6 + 16 + 7 \;=\; 29
$$

- 29 is close to 0 because nothing changes much across the window. Along the cat's edges the answers reach several hundred

**Reading the animation**

- The first three steps hold still long enough to check the sums. After the first row it speeds up
- Red: the image gets brighter from left to right there. Blue: it gets darker. Near white: no vertical edge
- Each answer is drawn where the centre of its window was, so the feature map lines up with the cat

In [ ]:
import torch.nn.functional as F
from matplotlib.patches import ConnectionPatch
from IPython.display import Image as NotebookImage                  # PIL's Image is already taken

kernel = np.array([[-1, 0, 1],
                   [-2, 0, 2],
                   [-1, 0, 1]])                                     # vertical edges: dark on the left, bright on the right
feature_map = F.conv2d(torch.tensor(grey, dtype=torch.float32)[None, None],     # (1 photo, 1 channel, 24, 24)
                       torch.tensor(kernel, dtype=torch.float32)[None, None])[0, 0].numpy()
H, W = feature_map.shape                                            # 24 - 3 + 1 = 22

by_hand = np.array([[(grey[i:i + 3, j:j + 3] * kernel).sum() for j in range(W)] for i in range(H)])
print(f"feature map {H} × {W}   multiply-and-add by hand equals F.conv2d at all {H * W} positions: "
      f"{np.array_equal(by_hand, feature_map)}")

# ---- one figure: the image, three 3 × 3 grids, the feature map ------------------------------------------
MARK, LIM = "#eda100", np.percentile(np.abs(feature_map), 98)       # LIM: where the map's colours max out
fig = plt.figure(figsize=(14, 4), dpi=72)
gs  = fig.add_gridspec(1, 5, width_ratios=[1.45, 1, 1, 1, 1.45], wspace=.32, left=.02, right=.98, top=.78, bottom=.12)
ax_img, ax_win, ax_ker, ax_prod, ax_map = map(fig.add_subplot, gs)

def grid(ax, title, cmap, lo, hi):                                  # a 3 × 3 matrix with its numbers written in
    ax.set_title(title, loc="left", weight="bold")
    return (ax.imshow(np.zeros((3, 3)), cmap=cmap, vmin=lo, vmax=hi),
            [ax.text(v, u, "", ha="center", va="center", fontsize=15) for u in range(3) for v in range(3)])

def fill(image, texts, M, dark):                                    # put new numbers into a grid
    image.set_data(M)
    for text, value in zip(texts, M.flat):
        text.set_text(f"{value:.0f}"); text.set_color("white" if dark(value) else "black")

ax_img.imshow(grey, cmap="gray", vmin=0, vmax=255)
ax_img.set_title(f"The image, {grey.shape[0]} × {grey.shape[1]}", loc="left", weight="bold")
win_im,  win_txt  = grid(ax_win,  "Pixels under the kernel", "gray", 0, 255)
ker_im,  ker_txt  = grid(ax_ker,  "Kernel", "RdBu_r", -2.6, 2.6)
prod_im, prod_txt = grid(ax_prod, "Pixel × weight", "RdBu_r", -660, 660)
fill(ker_im, ker_txt, kernel, lambda w: abs(w) == 2)
ax_ker.set_xlabel("the same 9 weights at every step", fontsize=11)
step  = ax_img.text(.5, -.08, "", transform=ax_img.transAxes, ha="center", va="top", fontsize=13)
total = ax_prod.text(.5, -.08, "", transform=ax_prod.transAxes, ha="center", va="top", fontsize=13, weight="bold")
for ax, symbol in ((ax_ker, "×"), (ax_prod, "=")):
    ax.text(-.17, .5, symbol, transform=ax.transAxes, ha="center", va="center", fontsize=30)
for ax in (ax_win, ax_prod):
    ax.spines[:].set(color=MARK, lw=3)

shown = np.full((H, W), np.nan)                                     # NaN: not computed yet, drawn grey
cmap  = plt.get_cmap("RdBu_r").copy(); cmap.set_bad("#dedede")
map_im = ax_map.imshow(shown, cmap=cmap, vmin=-LIM, vmax=LIM, extent=(.5, W + .5, H + .5, .5))
ax_map.set_xlim(-.5, W + 1.5); ax_map.set_ylim(H + 1.5, -.5)       # on the image's grid, so each answer sits
ax_map.spines[:].set_visible(False)                                 # where the centre of its window was
ax_map.set_title(f"Feature map, {H} × {W}", loc="left", weight="bold")
ax_map.set_xlabel("red: gets brighter left to right\nblue: gets darker left to right", fontsize=11)

box  = ax_img.add_patch(plt.Rectangle((-.5, -.5), 3, 3, fill=False, ec=MARK, lw=3, clip_on=False))
cell = ax_map.add_patch(plt.Rectangle((.5, .5), 1, 1, fill=False, ec=MARK, lw=3, clip_on=False))
zoom = [fig.add_artist(ConnectionPatch((0, 0), corner, ax_img.transData, ax_win.transAxes, color=MARK, lw=1.5))
        for corner in ((0, 1), (0, 0))]                             # from the box to the enlarged window
arrow = fig.add_artist(ConnectionPatch((1, .5), (0, 0), ax_prod.transAxes, ax_map.transData, color=MARK, lw=1.5,
                                       arrowstyle="-|>", mutation_scale=18, connectionstyle="arc3,rad=-.25"))
fig.text(.02, .91, "F.conv2d slides one 3 × 3 kernel over the image, one pixel at a time", fontsize=16, weight="bold")
for ax in (ax_img, ax_win, ax_ker, ax_prod, ax_map):
    ax.set_xticks([]); ax.set_yticks([])

# ---- the animation: draw what never changes once, then redraw only what moves ---------------------------
map_im.set_data(feature_map); fig.canvas.draw()                    # the finished picture, to pick 255 GIF colours
palette = Image.fromarray(np.asarray(fig.canvas.buffer_rgba())[..., :3]).quantize(255, dither=Image.Dither.NONE)

moving = [map_im, cell, win_im, *win_txt, prod_im, *prod_txt, *ax_win.spines.values(), *ax_prod.spines.values(),
          step, total, box, *zoom, arrow]                           # in drawing order, orange frames on top
for artist in moving:
    artist.set_animated(True)                                       # left out of fig.canvas.draw()
map_im.set_data(shown); fig.canvas.draw()
background = fig.canvas.copy_from_bbox(fig.bbox)

frames, durations = [], []
for k in range(H * W + 1):                                          # one frame per position, then the finished map
    i, j = divmod(min(k, H * W - 1), W)                             # row and column of the window's top-left pixel
    shown[i, j] = feature_map[i, j]
    map_im.set_data(shown)
    fill(win_im, win_txt, grey[i:i + 3, j:j + 3], lambda v: v < 110)
    fill(prod_im, prod_txt, grey[i:i + 3, j:j + 3] * kernel, lambda v: abs(v) > 300)
    total.set_text(f"add the nine:  {by_hand[i, j]}")
    step.set_text(f"step {k + 1} of {H * W}" if k < H * W else f"all {H * W} steps done")
    box.set_xy((j - .5, i - .5)); cell.set_xy((j + .5, i + .5))
    zoom[0].xy1, zoom[1].xy1 = (j + 2.5, i - .5), (j + 2.5, i + 2.5)
    arrow.xy2 = (j + .5, i + 1)
    for artist in (box, cell, *zoom, arrow):
        artist.set_visible(k < H * W)

    fig.canvas.restore_region(background)
    for artist in moving:
        fig.draw_artist(artist)
    frames.append(Image.fromarray(np.asarray(fig.canvas.buffer_rgba())[..., :3])
                  .quantize(palette=palette, dither=Image.Dither.NONE))
    durations.append(1500 if k < 3 else 200 if k < W else 40 if k < H * W else 3000)   # milliseconds on screen
plt.close(fig)

gif = DATA / "conv2d.gif"                                           # also a file, e.g. for slides
frames[0].save(gif, save_all=True, append_images=frames[1:], duration=durations, loop=0)
NotebookImage(filename=gif)